In [5]:
import os
import pandas as pd

# Function to delete a column and rename the file
def delete_column_and_rename(file_path, column_to_delete):
    # Extract the directory, filename, and extension
    directory, file_name = os.path.split(file_path)
    base_name, ext = os.path.splitext(file_name)
    
    # Load the CSV file into a DataFrame
    df = pd.read_csv(file_path)
    
    # Check if the column exists
    if column_to_delete in df.columns:
        # Drop the specified column
        df = df.drop(columns=[column_to_delete])
        print(f"Column '{column_to_delete}' deleted.")
    else:
        print(f"Column '{column_to_delete}' not found. No changes made.")
        return  # Exit if the column doesn't exist
    
    # Create the new file name with the suffix "_rev"
    new_file_name = f"{base_name}_rev{ext}"
    new_file_path = os.path.join(directory, new_file_name)
    
    # Save the modified DataFrame to a new CSV file
    df.to_csv(new_file_path, index=False)
    print(f"Modified file saved as: {new_file_path}")

# Example usage
file_path = "path_to_your_dataset.csv"  # Replace with the path to your dataset
column_to_delete = "column_name_to_delete"  # Replace with the column you want to delete

delete_column_and_rename("ds_235795_54.csv", "URLSimilarityIndex")


Column 'URLSimilarityIndex' deleted.
Modified file saved as: ds_235795_54_rev.csv


In [3]:
import os
import pandas as pd

# Function to move a column to the last position and rename the file
def move_column_to_last_and_rename(file_path, column_to_move):
    # Extract the directory, filename, and extension
    directory, file_name = os.path.split(file_path)
    base_name, ext = os.path.splitext(file_name)
    
    # Load the CSV file into a DataFrame
    df = pd.read_csv(file_path)
    
    # Check if the column exists
    if column_to_move in df.columns:
        # Move the column to the last position
        column_data = df.pop(column_to_move)  # Remove the column from its current position
        df[column_to_move] = column_data      # Add it to the end of the DataFrame
        print(f"Column '{column_to_move}' moved to the last position.")
    else:
        print(f"Column '{column_to_move}' not found. No changes made.")
        return  # Exit if the column doesn't exist
    
    # Create the new file name with the suffix "_rev"
    new_file_name = f"{base_name}_rev{ext}"
    new_file_path = os.path.join(directory, new_file_name)
    
    # Save the modified DataFrame to a new CSV file
    df.to_csv(new_file_path, index=False)
    print(f"Modified file saved as: {new_file_path}")

# Example usage
file_path = "path_to_your_dataset.csv"  # Replace with the path to your dataset
column_to_move = "column_name_to_move"  # Replace with the column you want to move

move_column_to_last_and_rename("ds_247950.csv", "Type")


Column 'Type' moved to the last position.
Modified file saved as: ds_247950_rev.csv


In [8]:
# Import necessary libraries
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
import pickle  # For saving the model

# Folder containing datasets
folder_path = "Dataset/"  # Replace with your folder path
model_name = "RandomForest"

# Loop through all datasets in the folder
for dataset_file in os.listdir(folder_path):
    if dataset_file.endswith(".csv"):  # Process only CSV files
        # Extract dataset name (without extension)
        dataset_name = os.path.splitext(dataset_file)[0]
        dataset_path = os.path.join(folder_path, dataset_file)
        
        print(f"Processing dataset: {dataset_name}")

        # Step 1: Load dataset
        df = pd.read_csv(dataset_path)

        # Step 2: Data preprocessing
        # Separate features (X) and target (y)
        X = df.iloc[:, :-1]  # All columns except the last one (features)
        y = df.iloc[:, -1]   # Last column (target)

        # Encode non-numeric columns
        for col in X.select_dtypes(include=['object']).columns:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col])

        # Encode target label if it is categorical
        if y.dtype == 'object':
            y = LabelEncoder().fit_transform(y)

        # Map class labels to ensure they are 0 and 1
        y = np.where(y == -1, 0, y)

        # Step 3: Split the data into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # Step 4: Handle imbalance in target using SMOTE
        smote = SMOTE(random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

        # Step 5: Train Random Forest model
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train_resampled, y_train_resampled)

        # Step 6: Save the trained model
        model_save_path = os.path.join(folder_path, f"{dataset_name}_{model_name}_NoFS.pkl")
        with open(model_save_path, "wb") as model_file:
            pickle.dump(rf_model, model_file)

        print(f"Model saved: {model_save_path}")

print("Processing complete for all datasets.")


Processing dataset: ds_100K20
Model saved: Dataset/ds_100K20_RandomForest_NoFS.pkl
Processing dataset: ds_10K18
Model saved: Dataset/ds_10K18_RandomForest_NoFS.pkl
Processing dataset: ds_10K50_rev
Model saved: Dataset/ds_10K50_rev_RandomForest_NoFS.pkl
Processing dataset: ds_11055
Model saved: Dataset/ds_11055_RandomForest_NoFS.pkl
Processing dataset: ds_11055_rev
Model saved: Dataset/ds_11055_rev_RandomForest_NoFS.pkl
Processing dataset: ds_11K89
Model saved: Dataset/ds_11K89_RandomForest_NoFS.pkl
Processing dataset: ds_129K112
Model saved: Dataset/ds_129K112_RandomForest_NoFS.pkl
Processing dataset: ds_235795_54_rev
Model saved: Dataset/ds_235795_54_rev_RandomForest_NoFS.pkl
Processing dataset: ds_247950_rev
Model saved: Dataset/ds_247950_rev_RandomForest_NoFS.pkl
Processing dataset: ds_600K11_rev
Model saved: Dataset/ds_600K11_rev_RandomForest_NoFS.pkl
Processing dataset: ds_88K112
Model saved: Dataset/ds_88K112_RandomForest_NoFS.pkl
Processing dataset: ds_90K32
Model saved: Dataset

In [9]:
# Import necessary libraries
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

# Directory containing datasets
input_folder = "dataset/"  # Replace with your folder path
output_csv = "modelRFNoFS_results.csv"

# Initialize a list to store results
results = []

# Loop through all files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):  # Process only CSV files
        dataset_path = os.path.join(input_folder, filename)
        
        # Load dataset
        print(f"Processing {filename}...")
        df = pd.read_csv(dataset_path)

        # Separate features (X) and target (y)
        X = df.iloc[:, :-1]
        y = df.iloc[:, -1]
        
        # Encode non-numeric columns
        for col in X.select_dtypes(include=['object']).columns:
            X[col] = pd.factorize(X[col])[0]
        
        # Encode target if it is categorical
        if y.dtype == 'object':
            y = pd.factorize(y)[0]

        # Handle imbalance in target using SMOTE
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X, y)

        # Split into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

        # Train Random Forest model
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train, y_train)

        # Evaluate the model
        y_pred = rf_model.predict(X_test)
        y_proba = rf_model.predict_proba(X_test)[:, 1]
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_test, y_proba)
        
        # Confusion matrix to calculate false positive rate
        conf_matrix = confusion_matrix(y_test, y_pred)
        false_positive_rate = conf_matrix[0, 1] / conf_matrix[0].sum() if conf_matrix[0].sum() > 0 else 0

        # Save results
        results.append({
            "Dataset Name": filename,
            "Number of Rows": df.shape[0],
            "Number of Columns": df.shape[1],
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "False Positive Rate": false_positive_rate,
            "ROC AUC": roc_auc
        })

# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)

print(f"Results saved to {output_csv}")


Processing ds_100K20.csv...
Processing ds_10K18.csv...
Processing ds_10K50_rev.csv...
Processing ds_11055.csv...
Processing ds_11055_rev.csv...
Processing ds_11K89.csv...
Processing ds_129K112.csv...
Processing ds_235795_54_rev.csv...
Processing ds_247950_rev.csv...
Processing ds_600K11_rev.csv...
Processing ds_88K112.csv...
Processing ds_90K32.csv...
Results saved to modelRFNoFS_results.csv


# RF no FS

In [10]:
# Import necessary libraries
import os
import pandas as pd
import numpy as np
import time  # Import time for measuring runtime
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

# Directory containing datasets
input_folder = "dataset/"  # Replace with your folder path
output_csv = "modelRFNoFS_results.csv"

# Initialize a list to store results
results = []

# Loop through all files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):  # Process only CSV files
        dataset_path = os.path.join(input_folder, filename)
        
        # Start time for the dataset
        start_time = time.time()

        # Load dataset
        print(f"Processing {filename}...")
        df = pd.read_csv(dataset_path)

        # Separate features (X) and target (y)
        X = df.iloc[:, :-1]
        y = df.iloc[:, -1]
        
        # Encode non-numeric columns
        for col in X.select_dtypes(include=['object']).columns:
            X[col] = pd.factorize(X[col])[0]
        
        # Encode target if it is categorical
        if y.dtype == 'object':
            y = pd.factorize(y)[0]

        # Handle imbalance in target using SMOTE
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X, y)

        # Split into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

        # Train Random Forest model
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train, y_train)

        # Evaluate the model
        y_pred = rf_model.predict(X_test)
        y_proba = rf_model.predict_proba(X_test)[:, 1]
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_test, y_proba)
        
        # Confusion matrix to calculate false positive rate
        conf_matrix = confusion_matrix(y_test, y_pred)
        false_positive_rate = conf_matrix[0, 1] / conf_matrix[0].sum() if conf_matrix[0].sum() > 0 else 0

        # End time for the dataset
        end_time = time.time()
        runtime = end_time - start_time

        # Save results
        results.append({
            "Dataset Name": filename,
            "Number of Rows": df.shape[0],
            "Number of Columns": df.shape[1],
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "False Positive Rate": false_positive_rate,
            "ROC AUC": roc_auc,
            "Runtime (seconds)": runtime
        })

# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)

print(f"Results saved to {output_csv}")


Processing ds_100K20_.csv...
Processing ds_10K18.csv...
Processing ds_10K50_rev.csv...
Processing ds_11055.csv...
Processing ds_11055_rev.csv...
Processing ds_11K89.csv...
Processing ds_129K112.csv...
Processing ds_235795_54_rev.csv...
Processing ds_247950_rev.csv...
Processing ds_600K11_rev.csv...
Processing ds_88K112.csv...
Processing ds_90K32.csv...
Results saved to modelRFNoFS_results.csv


# No FS XGB

In [13]:
# Import necessary libraries
import os
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# Directory containing datasets
input_folder = "dataset/"  # Replace with your folder path
output_csv = "modelXGBoost_results.csv"

# Initialize a list to store results
results = []

# Loop through all files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):  # Process only CSV files
        dataset_path = os.path.join(input_folder, filename)
        
        # Start time for the dataset
        start_time = time.time()

        # Load dataset
        print(f"Processing {filename}...")
        df = pd.read_csv(dataset_path)

        # Separate features (X) and target (y)
        X = df.iloc[:, :-1]
        y = df.iloc[:, -1]
        
        # Encode non-numeric columns
        for col in X.select_dtypes(include=['object']).columns:
            X[col] = pd.factorize(X[col])[0]
        
        # Encode target if it is categorical
        if y.dtype == 'object':
            y = pd.factorize(y)[0]

        # Map -1 to 0 to ensure binary class labels are 0 and 1
        y = np.where(y == -1, 0, y)

        # Check unique values in the target variable after mapping
        unique_classes = np.unique(y)
        print(f"Unique classes in target variable: {unique_classes}")

        # Ensure the unique classes are only [0, 1]
        if not np.array_equal(unique_classes, [0, 1]):
            raise ValueError(f"Unexpected classes in target variable: {unique_classes}. Expected [0, 1].")

        # Handle imbalance in target using SMOTE
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X, y)

        # Split into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

        # Train XGBoost model
        xgb_model = XGBClassifier(
            use_label_encoder=False,  # Suppress label encoding warning
            eval_metric='logloss',   # Set evaluation metric
            random_state=42
        )
        xgb_model.fit(X_train, y_train)

        # Evaluate the model
        y_pred = xgb_model.predict(X_test)
        y_proba = xgb_model.predict_proba(X_test)[:, 1]
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_test, y_proba)
        
        # Confusion matrix to calculate false positive rate
        conf_matrix = confusion_matrix(y_test, y_pred)
        false_positive_rate = conf_matrix[0, 1] / conf_matrix[0].sum() if conf_matrix[0].sum() > 0 else 0

        # End time for the dataset
        end_time = time.time()
        runtime = end_time - start_time

        # Save results
        results.append({
            "Dataset Name": filename,
            "Number of Rows": df.shape[0],
            "Number of Columns": df.shape[1],
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "False Positive Rate": false_positive_rate,
            "ROC AUC": roc_auc,
            "Runtime (seconds)": runtime
        })

# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)

print(f"Results saved to {output_csv}")


Processing ds_100K20_.csv...
Unique classes in target variable: [0 1]


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [21:52:15] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Processing ds_10K18.csv...
Unique classes in target variable: [0 1]


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [21:55:25] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Processing ds_10K50_rev.csv...
Unique classes in target variable: [0 1]


...

Processing ds_88K112.csv...
Unique classes in target variable: [0 1]


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [21:59:49] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Processing ds_90K32.csv...
Unique classes in target variable: [0 1]


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [21:59:50] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Results saved to modelXGBoost_results.csv


# No FS CatBoost

In [11]:
# Import necessary libraries
import os
import gc
import pandas as pd
import numpy as np
import time  # Import time for measuring runtime
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier

# Directory containing datasets
input_folder = "dataset/error/"  # Replace with your folder path
output_csv = "modelCatBoost_noFS_results.csv"

# Initialize a list to store results
results = []

# Loop through all files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):  # Process only CSV files
        dataset_path = os.path.join(input_folder, filename)
        
        # Start time for the dataset
        start_time = time.time()

        try:
            # Load dataset
            print(f"Processing {filename}...")
            df = pd.read_csv(dataset_path)

            # Separate features (X) and target (y)
            X = df.iloc[:, :-1]
            y = df.iloc[:, -1]
            
            # Detect categorical columns
            cat_features = X.select_dtypes(include=['object']).columns.tolist()

           
            
           
            # Encode target if it is categorical
            if y.dtype == 'object':
                y = pd.factorize(y)[0]
                # Map -1 to 0 to ensure binary class labels are 0 and 1
                y = np.where(y == -1, 0, y)

            # Encode non-numeric columns
            for col in X.select_dtypes(include=['object']).columns:
                le = LabelEncoder()
                X[col] = le.fit_transform(X[col])


            # Handle imbalance in target using SMOTE
            smote = SMOTE(random_state=42)
            X_resampled, y_resampled = smote.fit_resample(X, y)


            # Split into training and testing sets
            X_train, X_test, y_train, y_test = train_test_split(
                X_resampled, y_resampled, test_size=0.2, random_state=42
            )

            # Train CatBoost model
            catboost_model = CatBoostClassifier(
                verbose=0,  # Suppress output during training
                random_state=42
            )
            catboost_model.fit(X_train, y_train, cat_features=cat_features)

            # Evaluate the model
            y_pred = catboost_model.predict(X_test)
            y_proba = catboost_model.predict_proba(X_test)[:, 1]
            
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall = recall_score(y_test, y_pred, zero_division=0)
            roc_auc = roc_auc_score(y_test, y_proba)
            
            # Confusion matrix to calculate false positive rate
            conf_matrix = confusion_matrix(y_test, y_pred)
            false_positive_rate = conf_matrix[0, 1] / conf_matrix[0].sum() if conf_matrix[0].sum() > 0 else 0

            # End time for the dataset
            end_time = time.time()
            runtime = end_time - start_time

            # Save results
            results.append({
                "Dataset Name": filename,
                "Number of Rows": df.shape[0],
                "Number of Columns": df.shape[1],
                "Accuracy": accuracy,
                "Precision": precision,
                "Recall": recall,
                "False Positive Rate": false_positive_rate,
                "ROC AUC": roc_auc,
                "Runtime (seconds)": runtime
            })

        except Exception as e:
            print(f"Error processing {filename}: {e}")
        
        finally:
            # Check if variables exist before deleting them
            for var in ['df', 'X', 'y', 'X_resampled', 'y_resampled', 'X_train', 'X_test', 'y_train', 'y_test', 'catboost_model']:
                if var in locals():
                    del locals()[var]
            gc.collect()  # Force garbage collection

# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)

print(f"Results saved to {output_csv}")


Processing ds_10K18.csv...
Processing ds_11K89.csv...
Processing ds_235795_54_rev.csv...
Processing ds_90K32.csv...
Results saved to modelCatBoost_noFS_results.csv


# NoFS EBM

In [17]:
# Import necessary libraries
import os
import gc
import pandas as pd
import numpy as np
import time  # Import time for measuring runtime
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from interpret.glassbox import ExplainableBoostingClassifier
from imblearn.over_sampling import SMOTE

# Directory containing datasets
input_folder = "dataset/"  # Replace with your folder path
output_csv = "modelEBMNoFS_results.csv"

# Initialize a list to store results
results = []

# Loop through all files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):  # Process only CSV files
        dataset_path = os.path.join(input_folder, filename)
        
        # Start time for the dataset
        start_time = time.time()

        # Load dataset
        print(f"Processing {filename}...")
        df = pd.read_csv(dataset_path)

        # Separate features (X) and target (y)
        X = df.iloc[:, :-1]
        y = df.iloc[:, -1]
        
        # Encode non-numeric columns
        for col in X.select_dtypes(include=['object']).columns:
            X[col] = pd.factorize(X[col])[0]
        
        # Encode target if it is categorical
        if y.dtype == 'object':
            y = pd.factorize(y)[0]
            # Map -1 to 0 to ensure binary class labels are 0 and 1
            y = np.where(y == -1, 0, y)

        # Handle imbalance in target using SMOTE
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X, y)

        # Split into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

        # Train EBM model
        ebm_model = ExplainableBoostingClassifier(random_state=42)
        ebm_model.fit(X_train, y_train)

        # Evaluate the model
        y_pred = ebm_model.predict(X_test)
        y_proba = ebm_model.predict_proba(X_test)[:, 1]
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_test, y_proba)
        
        # Confusion matrix to calculate false positive rate
        conf_matrix = confusion_matrix(y_test, y_pred)
        false_positive_rate = conf_matrix[0, 1] / conf_matrix[0].sum() if conf_matrix[0].sum() > 0 else 0

        # End time for the dataset
        end_time = time.time()
        runtime = end_time - start_time

        # Save results
        results.append({
            "Dataset Name": filename,
            "Number of Rows": df.shape[0],
            "Number of Columns": df.shape[1],
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "False Positive Rate": false_positive_rate,
            "ROC AUC": roc_auc,
            "Runtime (seconds)": runtime
        })

# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)

print(f"Results saved to {output_csv}")


Processing ds_100K20_.csv...
Processing ds_10K18.csv...
Processing ds_10K50_rev.csv...
Processing ds_11055.csv...
Processing ds_11055_rev.csv...
Processing ds_11K89.csv...
Processing ds_129K112.csv...
Processing ds_235795_54_rev.csv...
Processing ds_247950_rev.csv...
Processing ds_600K11_rev.csv...
Processing ds_88K112.csv...
Processing ds_90K32.csv...
Results saved to modelEBM_results.csv
